# PICK: Semi-Supervised Segmentation for BHSD (Fold 0) on Kaggle GPU

### Hướng dẫn thiết lập môi trường Kaggle:
1. **Notebook Settings (Cột tùy chọn bên phải màn hình):**
   - **Accelerator**: Chọn **GPU T4 x 2** hoặc **GPU P100** (16GB VRAM).
   - **Internet**: Bật sang **ON** (bắt buộc: để clone mã nguồn GitHub, tải dataset từ Google Drive và cài đặt thư viện).
2. **Nguồn dữ liệu & Mã nguồn:**
   - **Mã nguồn**: Clone trực tiếp từ GitHub (`https://github.com/maxwell0027/PICK.git`).
   - **Dataset**: File `BHSD_fold0.tar` (~20GB) tải từ Google Drive bằng `gdown` và giải nén vào `/kaggle/working/BHSD_fold0`.

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q --upgrade gdown
!pip install -q medpy tensorboardX h5py SimpleITK nibabel

### Bước 2: Clone mã nguồn PICK từ GitHub
Mã nguồn sẽ được clone về `/kaggle/working/PICK`.

> **Lưu ý:** Nếu repository của bạn ở chế độ **Private**, hãy tạo **Personal Access Token (PAT)** trên GitHub (Settings -> Developer Settings -> Personal access tokens) và điền vào biến `GITHUB_TOKEN` bên dưới. Nếu repo là Public, bạn chỉ cần giữ nguyên.

In [ ]:
import os

# Cấu hình GitHub Repository
REPO_NAME = "maxwell0027/PICK"
GITHUB_TOKEN = ""  # Điền token ghp_xxx nếu repo là Private, nếu Public thì để trống

if GITHUB_TOKEN.strip():
    GITHUB_REPO_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_NAME}.git"
else:
    GITHUB_REPO_URL = f"https://github.com/{REPO_NAME}.git"

WORK_DIR = "/kaggle/working/PICK"

if not os.path.exists(WORK_DIR):
    print(f"[*] Đang clone repository {REPO_NAME}...")
    !git clone {GITHUB_REPO_URL} {WORK_DIR}
else:
    print(f"[OK] Repository đã tồn tại tại {WORK_DIR}. Đang cập nhật git pull...")
    !git -C {WORK_DIR} pull

%cd {WORK_DIR}
!git log -1 --oneline
!ls -la

### Bước 3: Tải Dataset BHSD Fold 0 từ Google Drive & Giải nén
Nhập **File ID** hoặc **Link chia sẻ Google Drive** của file `BHSD_fold0.tar`.

> **Hướng dẫn lấy Google Drive File ID:**
> 1. Trên Google Drive, chuột phải vào file `BHSD_fold0.tar` -> chọn **Chia sẻ (Share)** -> Đổi quyền thành **Bất kỳ ai có đường liên kết (Anyone with the link can view)**.
> 2. Link chia sẻ có dạng: `https://drive.google.com/file/d/1AbC-xYz_123456789/view?usp=sharing`
> 3. Phần nằm giữa `/d/` và `/view` chính là **File ID** (`1AbC-xYz_123456789`). Dán ID này vào `GDRIVE_FILE_ID` bên dưới.
>
> **Quản lý dung lượng đĩa Kaggle:**
> Ổ đĩa `/kaggle/working` có hạn mức ~73GB. File tar nặng 20GB và dữ liệu giải nén nặng 20GB. Sau khi giải nén xong, cell bên dưới sẽ **tự động xóa file `BHSD_fold0.tar`** để giải phóng ngay 20GB dung lượng cho model checkpoints.

In [ ]:
import os, glob, gdown

# ==============================================================================
# 1. ĐIỀN GOOGLE DRIVE FILE ID HOẶC LINK CỦA FILE BHSD_fold0.tar:
# ==============================================================================
GDRIVE_FILE_ID = "YOUR_GDRIVE_FILE_ID_HERE"  # Ví dụ: "1AbC-xYz_123456789"
GDRIVE_URL = ""  # Hoặc dán cả URL chia sẻ vào đây nếu muốn

TAR_DEST = "/kaggle/working/BHSD_fold0.tar"
EXTRACT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/working/BHSD_fold0"

# 2. Kiểm tra nếu thư mục giải nén đã đầy đủ thì bỏ qua tải
if os.path.exists(os.path.join(DATA_ROOT, "data")) and os.path.exists(os.path.join(DATA_ROOT, "splits")):
    print(f"[OK] Thư mục dataset đã tồn tại đầy đủ tại: {DATA_ROOT}")
else:
    # Nếu chưa có file tar, tiến hành tải từ Google Drive
    if not os.path.exists(TAR_DEST):
        print("[*] Đang tải BHSD_fold0.tar từ Google Drive...")
        if GDRIVE_URL.strip():
            gdown.download(url=GDRIVE_URL.strip(), output=TAR_DEST, fuzzy=True)
        elif GDRIVE_FILE_ID.strip() and GDRIVE_FILE_ID != "YOUR_GDRIVE_FILE_ID_HERE":
            gdown.download(id=GDRIVE_FILE_ID.strip(), output=TAR_DEST, quiet=False)
        else:
            # Fallback: kiểm tra xem có gắn qua Kaggle Dataset không
            kaggle_candidates = glob.glob("/kaggle/input/**/BHSD_fold0", recursive=True)
            if kaggle_candidates:
                DATA_ROOT = kaggle_candidates[0]
                print(f"[OK] Phát hiện dataset từ Kaggle Input: {DATA_ROOT}")
            else:
                raise ValueError("Vui lòng điền GDRIVE_FILE_ID hoặc GDRIVE_URL của file BHSD_fold0.tar!")

    # Giải nén và dọn dẹp file tar
    if os.path.exists(TAR_DEST):
        print(f"[*] Đang giải nén {TAR_DEST} vào {EXTRACT_DIR} (quá trình này mất ~2-3 phút)...")
        !tar -xf {TAR_DEST} -C {EXTRACT_DIR}
        print("[*] Giải nén hoàn tất! Đang xóa file tar để thu hồi 20GB bộ nhớ...")
        !rm -f {TAR_DEST}
        print(f"[OK] Dataset đã sẵn sàng tại: {DATA_ROOT}")

### Bước 4: Kiểm tra GPU & Tính toàn vẹn của Dataset (Integrity Check)
Ô này kiểm tra thông số phần cứng GPU và xác thực toàn bộ cấu trúc dữ liệu của Fold 0:
- Danh sách train: `splits/fold0_train.txt` (2,132 ca)
- Danh sách validation: `splits/fold0_val.txt` (38 ca)
- Danh sách labeled: `train_labeled.txt` (154 ca)
- Kho file volume HDF5: `data/*.h5` (đầy đủ 2,170 file)

In [ ]:
import torch, os, glob

print("=" * 60)
print("=== 1. KIỂM TRA PHẦN CỨNG GPU ===")
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Thiết bị:", torch.cuda.get_device_name(0))
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"Bộ nhớ VRAM: {vram_gb} GB")
else:
    print("[CẢNH BÁO] Không tìm thấy GPU! Hãy chọn GPU T4 hoặc P100 ở Notebook Settings.")

print("\n" + "=" * 60)
print("=== 2. KIỂM TRA TÍNH TOÀN VẸN CỦA DATASET FOLD 0 ===")
print("Đường dẫn DATA_ROOT:", DATA_ROOT)
assert os.path.exists(DATA_ROOT), f"Không tồn tại thư mục {DATA_ROOT}"

train_split = os.path.join(DATA_ROOT, "splits", "fold0_train.txt")
val_split = os.path.join(DATA_ROOT, "splits", "fold0_val.txt")
labeled_split = os.path.join(DATA_ROOT, "train_labeled.txt")
data_dir = os.path.join(DATA_ROOT, "data")

for p, name in [(train_split, "fold0_train.txt"), (val_split, "fold0_val.txt"), (data_dir, "data directory")]:
    assert os.path.exists(p), f"[LỖI] Thiếu {name} tại {p}"

with open(train_split) as f:
    train_cases = [line.strip() for line in f if line.strip()]
with open(val_split) as f:
    val_cases = [line.strip() for line in f if line.strip()]

labeled_count = 0
if os.path.exists(labeled_split):
    with open(labeled_split) as f:
        labeled_count = len([line.strip() for line in f if line.strip()])

h5_count = len(glob.glob(os.path.join(data_dir, "*.h5")))

print(f"[+] Số mẫu huấn luyện (train):      {len(train_cases)} (Kỳ vọng: 2132)")
print(f"[+] Số mẫu kiểm tra (val):          {len(val_cases)} (Kỳ vọng: 38)")
print(f"[+] Số mẫu có nhãn chuẩn (labeled): {labeled_count} (Kỳ vọng: 154)")
print(f"[+] Tổng số file .h5 trong data/:    {h5_count} (Kỳ vọng: 2170)")

# Xác nhận tính toàn vẹn mẫu ngẫu nhiên
sample_h5 = os.path.join(data_dir, f"{train_cases[0]}.h5")
assert os.path.exists(sample_h5), f"Không tìm thấy file mẫu {sample_h5}"

print("\n[OK] TẤT CẢ DỮ LIỆU ĐÃ ĐẦY ĐỦ VÀ SẴN SÀNG ĐỂ CHẠY HUẤN LUYỆN!")
print("=" * 60)

### Bước 5: Huấn luyện PICK - Binary Segmentation (Fold 0)

**Chi tiết tham số:**
- `--fold 0`: Chỉ định Fold 0 trong 5-fold cross validation.
- `--labelnum 154`: Sử dụng đầy đủ 154 ca có nhãn của Fold 0 (~7.2%). Nếu bạn muốn thử nghiệm bài toán ít nhãn hơn (ví dụ benchmark ~10% của tập labeled), bạn có thể đặt `--labelnum 15`.
- `--max_samples 2132`: Sử dụng toàn bộ 2,132 ca của tập train Fold 0.
- `--batch_size 4` & `--labeled_bs 2`: Mỗi batch gồm 2 ảnh labeled + 2 ảnh unlabeled (vừa vặn 16GB VRAM của T4/P100).
- `--pre_max_iteration 4000`: 4,000 bước pre-training.
- `--self_max_iteration 12000`: 12,000 bước self-training.
- Thời gian chạy: ~50-70 phút trên Kaggle GPU.

In [ ]:
# === HUẤN LUYỆN BINARY SEGMENTATION (FOLD 0) ===
FOLD_ID = 0
EXP_NAME = "BHSD_binary_PICK_fold0"
EXP_DIR = "/kaggle/working/experiments"

LABEL_NUM = 154       # Đầy đủ 154 ca có nhãn của Fold 0 (hoặc chọn 15 nếu muốn benchmark ít nhãn)
MAX_SAMPLES = 2132    # Toàn bộ 2132 ca train của Fold 0
BATCH_SIZE = 4
LABELED_BS = 2
PRE_ITERS = 4000
SELF_ITERS = 12000

!python BHSD_binary_PICK_train.py \
    --root_path {DATA_ROOT} \
    --exp {EXP_NAME} \
    --exp_dir {EXP_DIR} \
    --fold {FOLD_ID} \
    --model VNet \
    --labelnum {LABEL_NUM} \
    --max_samples {MAX_SAMPLES} \
    --batch_size {BATCH_SIZE} \
    --labeled_bs {LABELED_BS} \
    --pre_max_iteration {PRE_ITERS} \
    --self_max_iteration {SELF_ITERS} \
    --gpu 0

### Bước 6: (Tùy chọn) Huấn luyện PICK - Multi-class Segmentation (6 classes)
Phân đoạn 5 phân loại xuất huyết não chi tiết (EDH, IPH, IVH, SAH, SDH) + Background.
Bỏ comment ô dưới nếu muốn chạy huấn luyện Multi-class cho Fold 0.

In [ ]:
# === HUẤN LUYỆN MULTI-CLASS (FOLD 0) ===
# FOLD_ID = 0
# EXP_NAME = "BHSD_multiclass_PICK_fold0"
# EXP_DIR = "/kaggle/working/experiments"

# !python BHSD_multiclass_PICK_train.py \
#     --root_path {DATA_ROOT} \
#     --exp {EXP_NAME} \
#     --exp_dir {EXP_DIR} \
#     --fold {FOLD_ID} \
#     --model VNet \
#     --labelnum 154 \
#     --max_samples 2132 \
#     --batch_size 4 \
#     --labeled_bs 2 \
#     --pre_max_iteration 4000 \
#     --self_max_iteration 12000 \
#     --gpu 0

### Bước 7: Nén và Xuất Kết quả Model Checkpoints / Logs
Sau khi quá trình huấn luyện hoàn tất:
1. Ô này sẽ nén thư mục `/kaggle/working/experiments` (bao gồm model trọng số tốt nhất `VNet_best_model.pth`, log TensorBoard, và log huấn luyện `train.log`) thành file `bhsd_pick_fold0_results.tar.gz`.
2. Bạn có thể tải file này trực tiếp về máy tính tại mục **Output** (ở góc dưới bên phải giao diện Kaggle Notebook).

In [ ]:
import os

OUTPUT_ARCHIVE = "/kaggle/working/bhsd_pick_fold0_results.tar.gz"
EXP_DIR = "/kaggle/working/experiments"

if os.path.exists(EXP_DIR):
    print("[*] Đang nén toàn bộ kết quả mô hình và logs...")
    !tar -czvf {OUTPUT_ARCHIVE} -C {EXP_DIR} .
    size_mb = round(os.path.getsize(OUTPUT_ARCHIVE) / (1024**2), 2)
    print(f"\n[OK] Đã nén thành công file kết quả: {OUTPUT_ARCHIVE} ({size_mb} MB)")
    print("[*] File đã sẵn sàng tải về tại tab Output bên phải màn hình!")
else:
    print(f"[!] Chưa tìm thấy thư mục {EXP_DIR}. Vui lòng kiểm tra lại quá trình huấn luyện.")